# Run-book 209 — does rise-eo still work against Planetary Computer's API?

**What this proves:** every part of MPC that rise-eo's MPC source depends on still behaves as rise-eo
expects (issue #209). A downstream tool reported on 2026-11-09 that MPC changed its API without notice.
rise-eo uses the STAC search (with the `query` extension for cloud cover), SAS signing through the
`planetary-computer` package, a few item properties, the asset keys, and blob reads. It does **not**
use MPC's Data API (tiles, mosaics, previews).

**Claude does not run this.** You run it cell by cell and paste back the **Summary** cell's output, or
a failing cell's message. Every check is a plain `assert` you can read: the text above a check cell
says what PASS means, and the `assert` says the same thing in code.

**Needs:** network; no credentials (MPC is anonymous). Steps 1–2 are searches (seconds each).
Step 3 downloads one granule's B04 and SCL: about 100–250 MB (estimate) and under a minute on a
normal connection.

## Before you start

**Kernel — which Python runs the cells.** Use **`.venv`**: the Python at `rise-eo/.venv/bin/python`
in the **main** checkout (Python 3.11). It is the only environment with rise-eo's dependencies.
- Open VS Code on the **`rise-eo/` folder** (the main checkout) — even when this notebook lives in a
  worktree under `rise-eo/.claude/worktrees/<name>/`. Then **Select Kernel** (top right) →
  **Python Environments** → **`.venv`**.
- Not listed (e.g. VS Code was opened on the worktree folder, which has no `.venv` of its own)?
  **Select Another Kernel… → Python Environments → Enter interpreter path…** (wording varies by
  VS Code version) and give the main checkout's `rise-eo/.venv/bin/python`.
- The **Setup** cell below checks this and stops with a message if the kernel is wrong.

**Code — which `rise-eo` gets imported.** rise-eo is installed from the *main* checkout, so inside a
worktree a plain `import rise_eo` would quietly run `main`'s code, not the branch under test. The
Setup cell puts **this notebook's own** `src/` first on Python's import path and checks it worked.
(This is what the old run-books did with `export PYTHONPATH=src` — the same thing, invisibly.)

**Stop / resume.** Interrupt the kernel (■) to stop. Re-running a cell resumes: downloads skip
files that are already complete. After a kernel restart, run **Setup** and **Settings** again,
then continue at any step — every step reads what it needs from disk.

**When you are done.** Paste back what the **Summary** cell says to paste (plus any failing cell's
message). Then **Restart & Clear All Outputs** and save: this file is tracked in git, and a test
refuses saved outputs.

In [ ]:
# Setup -- which Python, and which rise-eo. Nothing to change here.
import sys
from pathlib import Path

# 1. Which Python? It must be rise-eo's own virtual environment, rise-eo/.venv.
VENV = Path(sys.prefix)
assert VENV.name == ".venv" and (VENV.parent / "pyproject.toml").exists(), (
    f"Wrong kernel: {sys.prefix} is not rise-eo's .venv. Pick the '.venv' kernel (see above).")
MAIN_CHECKOUT = VENV.parent              # the rise-eo checkout that owns the venv

# 2. Which rise-eo? The one next to this notebook: <checkout>/runbooks/<this> -> <checkout>/src.
#    Jupyter and VS Code start the kernel in the notebook's own folder.
CHECKOUT = Path.cwd().parent
assert Path.cwd().name == "runbooks" and (CHECKOUT / "src" / "rise_eo").is_dir(), (
    f"The kernel started in {Path.cwd()}, not in this notebook's runbooks/ folder.")
if str(CHECKOUT / "src") not in sys.path:
    sys.path.insert(0, str(CHECKOUT / "src"))
import rise_eo

assert Path(rise_eo.__file__).is_relative_to(CHECKOUT / "src"), (
    f"rise-eo was already imported from {rise_eo.__file__}. Restart the kernel, then run this cell first.")
print("Python :", sys.executable)
print("rise-eo    :", rise_eo.__file__)

## Settings

Every input this run-book uses. Change them here and nowhere else.

In [ ]:
# Settings
import datetime

OUT = MAIN_CHECKOUT / "tests" / "outputs" / "209_mpc_api_check"   # gitignored; outlives a worktree
ROI = CHECKOUT / "notebooks" / "shapefiles" / "s2grid=4772924.geojson"   # inside T33UWP
# The tracked tutorial fixture: 36 MPC granules over this ROI, April-September 2018. Its `id`
# column holds MPC item ids (issue #121), and it records each item's cloud cover.
FIXTURE_CATALOG = CHECKOUT / "tests" / "data" / "tutorial" / "sentinel-2-l2a" / "catalog.parquet"
FIXTURE_START = datetime.datetime(2018, 4, 1)
FIXTURE_END = datetime.datetime(2018, 10, 1)
MAX_CLOUDCOVER = 30                                  # any value that splits the 2018 window
# A recent window: whatever MPC serves today, after the reported change.
RECENT_START = datetime.datetime(2026, 10, 1)
RECENT_END = datetime.datetime(2026, 11, 9)
S2 = "sentinel-2-l2a"
S1 = "sentinel-1-rtc"                                # search only: one S1 scene is ~3.7 GB
BANDS = ["B04", "SCL"]                               # SCL is small; B04 shows the pixel range

## Step 1 — the 2018 search still returns the fixture's items, with the same cloud cover

**What it does:** searches MPC over the fixture's window with no cloud filter (latest
processing per acquisition, the default), and again with `max_cloudcover=MAX_CLOUDCOVER`. Matches each fixture
item to a live row by the MPC item id at the end of the row's `s3url` (the item's STAC link). Writes
`OUT/step1.json`.
**PASS if:** every fixture item comes back (if one is missing, MPC may have reprocessed it: compare
its id with the live ids before calling it a break); each one's cloud cover equals the fixture's; the
filtered search is non-empty, smaller than the unfiltered one, with every cloud cover at most
`MAX_CLOUDCOVER`; and a raw `pystac_client` search sending the same STAC `query` rise-eo sends gets
the same kind of answer from MPC itself (rise-eo also filters on the client, which would hide MPC
ignoring `query`).
**Time / disk:** seconds; one small file.

In [ ]:
# Step 1 -- two searches over the fixture's window.
import json
from pathlib import PurePosixPath

import geopandas as gpd
import pystac_client

from rise_eo import config
from rise_eo.sources import mpc

OUT.mkdir(parents=True, exist_ok=True)
everything = mpc.query_catalog(str(ROI), FIXTURE_START, FIXTURE_END, max_cloudcover=None, collection=S2)
filtered = mpc.query_catalog(str(ROI), FIXTURE_START, FIXTURE_END, max_cloudcover=MAX_CLOUDCOVER,
                             collection=S2)
live_cloud = {PurePosixPath(u).name: float(c) for u, c in zip(everything["s3url"], everything["cloud_cover"])}
fixture = gpd.read_parquet(FIXTURE_CATALOG)
# query_catalog also filters cloud cover on the client, so it would pass even if MPC ignored
# `query`. This raw search, with the same `query` rise-eo sends, shows what MPC itself returns.
raw = pystac_client.Client.open(config.MPC_STAC_URL).search(
    collections=[S2], datetime=[FIXTURE_START, FIXTURE_END],
    intersects=gpd.read_file(ROI).to_crs("EPSG:4326").geometry.union_all(),
    query={"eo:cloud_cover": {"lte": MAX_CLOUDCOVER}},
)
raw_cloud = [float(it.properties["eo:cloud_cover"]) for it in raw.items()]
(OUT / "step1.json").write_text(json.dumps({
    "n_live": len(everything),
    "n_filtered": len(filtered),
    "filtered_max_cloud": float(filtered["cloud_cover"].max()) if len(filtered) else None,
    "n_raw_query": len(raw_cloud),
    "raw_query_max_cloud": max(raw_cloud, default=None),
    "fixture": {i: float(c) for i, c in zip(fixture["id"], fixture["cloud_cover"])},
    "live": live_cloud,
}, indent=1))
print(len(everything), "live items;", len(filtered), "at most", MAX_CLOUDCOVER, "% cloud;",
      len(fixture), "fixture items")

In [ ]:
# Step 1 -- check it.
s1 = json.loads((OUT / "step1.json").read_text())
missing = sorted(set(s1["fixture"]) - set(s1["live"]))
assert not missing, f"expected every fixture item in the live search, missing {len(missing)}: {missing[:3]}"
changed = {i: (c, s1["live"][i]) for i, c in s1["fixture"].items() if abs(c - s1["live"][i]) > 1e-6}
assert not changed, f"expected the fixture's cloud cover, found (fixture, live): {changed}"
assert 0 < s1["n_filtered"] < s1["n_live"], (
    f"expected the cloud filter to keep some but not all items, kept {s1['n_filtered']} of {s1['n_live']}")
assert s1["filtered_max_cloud"] <= MAX_CLOUDCOVER, (
    f"expected cloud cover at most {MAX_CLOUDCOVER}, found {s1['filtered_max_cloud']}")
assert 0 < s1["n_raw_query"] < s1["n_live"], (
    f"expected MPC's own `query` filter to drop some items, it returned {s1['n_raw_query']} of {s1['n_live']}")
assert s1["raw_query_max_cloud"] <= MAX_CLOUDCOVER, (
    f"expected MPC to return cloud cover at most {MAX_CLOUDCOVER}, found {s1['raw_query_max_cloud']}")
print("PASS", len(s1["fixture"]), "fixture items found;", s1["n_filtered"], "of", s1["n_live"], "kept by the filter")

## Step 2 — recent items still parse: Sentinel-2 names and processing, Sentinel-1 orbit state

**What it does:** searches the recent window for Sentinel-2 (latest processing only) and
Sentinel-1 RTC, with no download. Building each row already parses the item: a Sentinel-2 row's
canonical name and processing columns come from `s2:product_uri`, `s2:processing_baseline` and
`s2:generation_time`. Writes the Sentinel-2 rows to `OUT/recent_s2.parquet` and a summary to
`OUT/step2.json`.
**PASS if:** both searches return items; no Sentinel-2 row lacks `processing_version`,
`processing_datetime` or `cloud_cover`; every Sentinel-1 item's `sat:orbit_state` is `ascending` or `descending`.
**Time / disk:** seconds; two small files.

In [ ]:
# Step 2 -- recent Sentinel-2 and Sentinel-1 searches.
recent_s2 = mpc.query_catalog(str(ROI), RECENT_START, RECENT_END, collection=S2)
recent_s1 = mpc.query_catalog(str(ROI), RECENT_START, RECENT_END, collection=S1)
recent_s2.to_parquet(OUT / "recent_s2.parquet")
(OUT / "step2.json").write_text(json.dumps({
    "n_s2": len(recent_s2),
    "s2_nulls": {c: int(recent_s2[c].isna().sum())
                 for c in ["processing_version", "processing_datetime", "cloud_cover"]},
    "n_s1": len(recent_s1),
    # `properties` holds each item's STAC properties as a JSON string.
    "s1_without_orbit_state": int(sum(json.loads(p).get("sat:orbit_state") not in ("ascending", "descending")
                                      for p in recent_s1["properties"])),
    "newest_s2": recent_s2.sort_values("timestamp")["id"].iloc[-1] if len(recent_s2) else None,
}, indent=1))
print(len(recent_s2), "Sentinel-2 items;", len(recent_s1), "Sentinel-1 items")

In [ ]:
# Step 2 -- check it.
s2 = json.loads((OUT / "step2.json").read_text())
assert s2["n_s2"] > 0, "expected Sentinel-2 items in the recent window, found none"
assert not any(s2["s2_nulls"].values()), f"expected no empty columns, found {s2['s2_nulls']}"
assert s2["n_s1"] > 0, "expected Sentinel-1 items in the recent window, found none"
assert s2["s1_without_orbit_state"] == 0, (
    f"expected sat:orbit_state on every Sentinel-1 item, missing on {s2['s1_without_orbit_state']}")
print("PASS", s2)

## Step 3 — sign and download the newest Sentinel-2 granule, then check its pixels

**What it does:** downloads B04 and SCL of the newest Sentinel-2 granule from Step 2 with
`rise_eo.download` (signing per transfer, then reading the signed blob). Reads each file back at
1/10 resolution (from its overviews).
**PASS if:** the download writes a catalog row and both files; B04 is `uint16` with a maximum
above 255 (a uint8 truncation like MPC's October incident would cap it at 255); SCL holds only
classes 0–11.
**Time / disk:** under a minute; about 100–250 MB (estimate) in `OUT/archive`.

In [ ]:
# Step 3 -- download the newest granule's bands.
import pandas as pd

import rise_eo

newest = pd.read_parquet(OUT / "recent_s2.parquet").sort_values("timestamp").iloc[-1]
day = newest["timestamp"].to_pydatetime().replace(hour=0, minute=0, second=0, microsecond=0, tzinfo=None)
catalog_filepath = rise_eo.download(
    roi=str(ROI), startdate=day, enddate=day + datetime.timedelta(hours=23, minutes=59), bands=BANDS,
    dst_folderpath=str(OUT / "archive"), source="mpc", collection=S2, max_tiles=1,
)
print("newest:", newest["id"], "| catalog:", catalog_filepath)

In [ ]:
# Step 3 -- check it: read the catalog and both files back from disk.
import numpy as np
import rasterio


def read_decimated(path):
    """(dtype, array) of band 1 at 1/10 resolution, read from the COG's overviews."""
    with rasterio.open(path) as src:
        return src.dtypes[0], src.read(1, out_shape=(src.height // 10, src.width // 10))


newest = pd.read_parquet(OUT / "recent_s2.parquet").sort_values("timestamp").iloc[-1]
catalog = gpd.read_parquet(rise_eo.archive_catalog_filepath(str(OUT / "archive"), S2))
assert newest["id"] in set(catalog["id"]), f"expected {newest['id']} in the catalog, found {list(catalog['id'])}"
found = {p.stem: p for p in (OUT / "archive").rglob("*.tif") if newest["id"] in str(p)}
assert set(BANDS) <= set(found), f"expected files {BANDS}, found {sorted(found)}"
b04_dtype, b04 = read_decimated(found["B04"])
_, scl = read_decimated(found["SCL"])
assert b04_dtype == "uint16", f"expected B04 as uint16, found {b04_dtype}"
assert int(b04.max()) > 255, f"expected B04 values above 255, found a maximum of {int(b04.max())}"
assert set(np.unique(scl).tolist()) <= set(range(12)), f"expected SCL classes 0-11, found {np.unique(scl)}"
(OUT / "step3.json").write_text(json.dumps({
    "granule": newest["id"], "b04_max": int(b04.max()), "scl_classes": sorted(np.unique(scl).tolist()),
}))
print("PASS", (OUT / "step3.json").read_text())

## Summary — paste this output back

Re-reads everything from disk, so it works after a restart. A step that did not run shows `null`.

In [ ]:
import json
from importlib.metadata import version


def read_step(name):
    """A step's saved JSON, or None if that step has not run."""
    path = OUT / f"{name}.json"
    return json.loads(path.read_text()) if path.exists() else None


step1 = read_step("step1")
summary = {
    "rise_eo": rise_eo.__version__,
    "planetary_computer": version("planetary-computer"),
    "pystac_client": version("pystac-client"),
    "step1": step1 and {k: step1.get(k) for k in ["n_live", "n_filtered", "filtered_max_cloud",
                                                   "n_raw_query", "raw_query_max_cloud"]}
             | {"fixture_items_found": len(set(step1["fixture"]) & set(step1["live"])),
                "fixture_items": len(step1["fixture"])},
    "step2": read_step("step2"),
    "step3": read_step("step3"),
}
print(json.dumps(summary, indent=2, default=str))